# Day 14: Production RAG Pipeline

We've learned the pieces. Today, we put them together.

**PDF → Chunking → Vector DB → Retrieval → LLM** — a complete RAG system.

## Install PDF Library

In [ ]:
%pip install PyPDF2 -q

## Setup

In [49]:
from google import genai
import chromadb
from PyPDF2 import PdfReader
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

## Step 1: Load PDF

In [50]:
# Load PDF
pdf_path = "yash.pdf"  # Your PDF file
reader = PdfReader(pdf_path)

# Extract text from all pages
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print(f"📄 Loaded PDF: {len(reader.pages)} pages, {len(full_text)} characters")
print(f"\n📖 Preview:\n{full_text[:500]}...")

📄 Loaded PDF: 2 pages, 2709 characters

📖 Preview:
  ContactBangalore, Indiawww.linkedin.com/in/yashwanth-s-725545213 (LinkedIn)Top SkillsGenAILarge Language Models (LLM)AI PromptingCertificationsPython Data StructuresProgramming for Everybody (GettingStarted with Python)Problem Solving(Basics)Yashwanth SDeveloper Associate @ SAP Labs India | MTech @ BITS PilaniBengaluru, Karnataka, IndiaSummaryI believe in delivering more than just technical solutions—my focusis on creating data-driven insights, seamless user experiences,and innovative AI-power...


## Step 2: Chunk the Document

In [53]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
    return chunks

chunks = chunk_text(full_text)

print(f"✂️ Created {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i}: {chunk[:80]}...\n")

✂️ Created 7 chunks

Chunk 0: ContactBangalore, Indiawww.linkedin.com/in/yashwanth-s-725545213 (LinkedIn)Top S...

Chunk 1: seamless user experiences,and innovative AI-powered tools that help businesses g...

Chunk 2: my work is rootedin delivering results that matter.My technical proficiency span...



## Step 3: Store in Vector Database

In [54]:
# Create ChromaDB collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="pdf_store2")

# Generate embeddings and store
for i, chunk in enumerate(chunks):
    embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    ).embeddings[0].values
    
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[f"chunk_{i}"]
    )

print(f"✅ Stored {collection.count()} chunks in ChromaDB")

✅ Stored 7 chunks in ChromaDB


## Step 4: The RAG Function

In [55]:
def ask(question):
    """Complete RAG pipeline: retrieve + generate."""
    
    # Embed the question
    query_emb = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    ).embeddings[0].values
    
    # Retrieve relevant chunks
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=3
    )
    context = "\n\n".join(results['documents'][0])
    
    # Generate answer
    prompt = f"""Answer the question based ONLY on the context below.
If the answer is not in the context, say "I couldn't find that information."

Context:
{context}

Question: {question}

Answer:"""
    
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )
    
    return response.text, results['documents'][0]

## Test: Ask Questions About the PDF

In [56]:
question = "what is Yash educational background?"

answer, sources = ask(question)

print(f"❓ {question}\n")
print(f"💬 {answer}\n")
print("📄 Sources:")
for i, src in enumerate(sources[1:]):
    print(f"  [{i+1}] {src[:100]}...")

❓ what is Yash educational background?

💬 BITS Pilani Work Integrated Learning Programmes - Master of Technology - MTech, Computer Software Engineering (2023 - 2025)
Global Academy Of Technology - Bachelor's degree, Computer Science (2019 - 2023)

📄 Sources:
  [1] and validation processes for critical quarterly changes.- Delivered high-priority production fixes a...
  [2] ngaluru, Karnataka, IndiaAI Solutions Development Engineer June 2024 - March 2025 (10 months)Bengalu...


In [57]:
# Try your own questions!
question = input("Ask a question: ")
answer, sources = ask(question)
print(f"\n💬 {answer}")


💬 GenAI, Large Language Models (LLM), AI Prompting, Python, SQL, and JavaScript.


## The Complete Pipeline

```
PDF → Extract Text → Chunk → Embed → Store in Vector DB
                                        ↓
Question → Embed → Search → Retrieve → LLM → Answer
```

## Key Takeaways

1. **Extract** text from PDFs using PyPDF2
2. **Chunk** with overlap to preserve context
3. **Store** embeddings in a vector database
4. **Retrieve** and **Generate** — that's RAG

---

This wraps up RAG basics. **Next section: Agents** — when retrieval isn't enough and you need to take actions.